# Data0 v2: three-way split and local m6Anet training

This notebook reads the logged one-epoch CPU experiment. It does not retrain or evaluate test data.

- Fold 0 remains validation. Fold 1 is now test; all other folds are training.
- Gene groups and transcripts are disjoint. Normalization uses training reads only.
- New test sites appeared in historical v1 training. Use newly fitted v2 models for subsequent evaluation.
- One epoch checks feasibility and timing; it does not establish converged accuracy.
- Historical pretrained and classical results use more training data or pretrained weights and are not matched v2 training comparisons.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
from m6a_project.splitting import assert_disjoint

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
summary = json.loads((ROOT / "docs/results/m6anet_training_smoke.json").read_text())
run = ROOT / summary["run_path"]
split = pd.read_csv(run / "split.csv")
assert_disjoint(split)
assert summary["metrics"]["test_evaluated"] is False
display(pd.DataFrame(summary["split"]).T)
print(run)

,sites,genes,positive_sites,prevalence
test,24694.0,770.0,1107.0,0.044829
train,71794.0,2307.0,3265.0,0.045477
val,25350.0,775.0,1103.0,0.043511


/Users/ncduy/Git/dsa4262-project/experiments/data0_m6anet_scratch_v2/20261005T103446_official_m6anet_m6anet_scratch_119c482f


In [2]:
timing = summary["timing"]
display(pd.Series({
    "Input adaptation (seconds)": summary["metrics"]["adapter_seconds"],
    "Train-only normalization (seconds)": timing["normalization_seconds"],
    "One training epoch (seconds)": timing["train_seconds"],
    "Five validation passes (seconds)": timing["validation_seconds"],
    "Total run (seconds)": summary["metrics"]["total_seconds"],
    "Main process peak RSS (MiB)": timing["main_process_peak_rss_mib"],
    "Largest child peak RSS (MiB)": timing["largest_child_peak_rss_mib"],
}))
display(pd.Series(summary["metrics"])[["average_precision", "roc_auc", "pr_auc_trapezoid", "brier_score"]])

Input adaptation (seconds)             39.844402
Train-only normalization (seconds)     64.935482
One training epoch (seconds)          115.720348
Five validation passes (seconds)      109.412347
Total run (seconds)                   335.138237
Main process peak RSS (MiB)           390.625000
Largest child peak RSS (MiB)          285.730469
dtype: float64

average_precision    0.367761
roc_auc              0.865749
pr_auc_trapezoid     0.367181
brier_score          0.144862
dtype: object

In [3]:
predictions = pd.read_parquet(run / "validation_predictions.parquet")
expected = split.query("partition == 'val'")
keys = ["transcript_id", "transcript_position"]
assert set(map(tuple, predictions[keys].to_numpy())) == set(map(tuple, expected[keys].to_numpy()))
assert set(predictions.partition) == {"val"}
display(predictions.head())

,transcript_id,transcript_position,n_reads,start,end,sequence,gene_id,label,partition,set_type,score,sampling_std
0,ENST00000000412,355,50,206793,209801,GAAACTA,ENSG00000003056,0,val,Val,0.037933,0.019978
1,ENST00000000412,367,47,209801,212634,GGGACCG,ENSG00000003056,0,val,Val,0.481230,0.046467
2,ENST00000000412,496,51,212634,215696,AGGACTG,ENSG00000003056,0,val,Val,0.233432,0.074755
3,ENST00000000412,501,56,215696,219051,TGGACTG,ENSG00000003056,0,val,Val,0.439397,0.127611
4,ENST00000000412,547,52,219051,222162,CAGACAG,ENSG00000003056,0,val,Val,0.649734,0.063136


To reproduce the logged experiment, use the command in its `rerun.txt`. The main entry point is `uv run python scripts/train_m6anet.py --config configs/experiments/m6anet_train.toml`. Settings are in `configs/experiments/m6anet_train.toml`.

For matched classical comparisons, run the default v2 experiment configuration. Choose a model using validation before explicitly evaluating the held-out test fold. Data1 and data2 remain untouched.